# White Castle rules assistant

RAG agent that answers questions about the rules of the board game White Castle:
- Knowledge base: the Spanish rulebook, indexed in ChromaDB with Gemini embeddings
- Agent: LangGraph graph (retrieve → generate) with Gemini and conversation memory

In [ ]:
# Imports
import base64
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from pypdf import PdfReader

In [ ]:
# Paths
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
PDF_PATH = PROJECT_DIR / "data" / "white_castle.pdf"
TRANSCRIPT_DIR = PROJECT_DIR / "data" / "transcript"

# Models
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"

## Setup

Loads the Gemini API key from `.env` (variable `GOOGLE_API_KEY`). The key is never printed.

In [3]:
def load_api_key():
    """Gemini API key: .env → environment variable GOOGLE_API_KEY."""
    load_dotenv(ENV_PATH)
    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError(f"GOOGLE_API_KEY is empty or missing in {ENV_PATH}")


load_api_key()

## Smoke test

Checks that both models answer before building anything on top of them.

In [4]:
def check_chat():
    """Chat model check: short prompt → printed reply."""
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    reply = llm.invoke("Reply in one short sentence: are you working?")
    print(f"{CHAT_MODEL}: {reply.text}")


def check_embeddings():
    """Embedding model check: short text → printed vector size."""
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
    vector = embeddings.embed_query("White Castle")
    print(f"{EMBEDDING_MODEL}: vector of size {len(vector)}")


check_chat()
check_embeddings()

gemini-3.5-flash-lite: Yes, I am currently active and ready to assist you.
gemini-embedding-2: vector of size 3072


## 1. Transcription

The rulebook PDF is scanned: every page is a single image with no text layer. Gemini reads each page image and transcribes it:
- One request per page → one Markdown file per page in `data/transcript/`
- Pages already transcribed are skipped, so the quota is only spent once
- Icons and illustrated examples that carry rules are described briefly between square brackets

In [ ]:
# Constants
TRANSCRIPTION_PROMPT = """You are transcribing one page of the Spanish rulebook of the board game White Castle.
- Transcribe all the text literally, in Spanish. Do not translate, summarize or correct it.
- Mark section titles with Markdown headings: # for main titles, ## for subsections.
- When an icon, diagram or illustrated example carries rules information, describe it briefly in Spanish between square brackets, for example: [Icono: 2 monedas].
- Ignore purely decorative illustrations, page numbers and repeated headers or footers.
- Return only the transcription, with no comments before or after it."""

In [ ]:
def transcribe_manual():
    """Rulebook PDF → one Markdown file per page in TRANSCRIPT_DIR (pages already done are skipped)."""
    TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    pages = PdfReader(PDF_PATH).pages
    for number, page in enumerate(pages, start=1):
        transcribe_page(llm, page, number)


def transcribe_page(llm, page, number):
    """One page: scanned image → Gemini transcription → page_NN.md."""
    output_path = TRANSCRIPT_DIR / f"page_{number:02d}.md"
    if output_path.exists():
        print(f"Page {number:02d}: already transcribed, skipped")
        return
    message = build_page_message(page)
    reply = llm.invoke([message])
    output_path.write_text(reply.text, encoding="utf-8")
    print(f"Page {number:02d}: transcribed ({len(reply.text)} characters)")


def build_page_message(page):
    """Message for Gemini: transcription prompt + the page JPEG image as base64."""
    image_b64 = base64.b64encode(page.images[0].data).decode()
    return HumanMessage(content=[
        {"type": "text", "text": TRANSCRIPTION_PROMPT},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
    ])


transcribe_manual()